# Сколько образцов выдержали проверку?

Две лаборатории испытывают упаковку на герметичность. Один объект будет хранить итоги нескольких проверок одной партии. Задача программы — принимать протоколы, отклонять невозможные записи и выдавать проверяемый итог по партии.

Партия K-17 получает сначала протокол «5 проверено, 1 протекает», затем «3 проверено, 2 протекают». У партии L-02 отдельный протокол «4 проверено, 0 протекает». Все числа в протоколах целые. Запись допустима, когда проверено хотя бы одно изделие, а число протечек находится между нулём и количеством проверенных включительно.

Выполняйте ячейки по порядку. Перед запуском попробуйте предсказать вывод.

## Шаг 1. Как записать первую проверку так, чтобы показатели двух партий хранились независимо?

In [ ]:
class Batch:
    def __init__(self, label):
        self.label = label
        self.tested = 0
        self.leaks = 0

    def record(self, tested, leaks):
        if tested <= 0 or leaks < 0 or leaks > tested:
            raise ValueError("невозможный протокол")
        self.tested += tested
        self.leaks += leaks

    def passed(self):
        return self.tested - self.leaks

first = Batch("K-17")
second = Batch("L-02")
first.record(5, 1)
print(first.label, first.tested, first.leaks)
print(second.label, second.tested, second.leaks)

**Обсуждение.** Метод record увеличивает счётчики только того экземпляра, у которого его вызвали. В первой партии уже испытали пять упаковок. У второй партии ещё нет результатов, поэтому оба её счётчика по-прежнему равны нулю.

## Шаг 2. Что останется в партии, если новый протокол обещает три протечки в двух проверках?

In [ ]:
before = (first.tested, first.leaks)
try:
    first.record(2, 3)
except ValueError as error:
    print(error)
print(before, (first.tested, first.leaks))

**Обсуждение.** Из двух упаковок невозможно получить три протекающие упаковки, если каждую испытывают один раз. Метод проверяет оба числа до изменения счётчиков. Сравнение состояния до и после отказа подтверждает, что в отчёт не попало даже частичное обновление.

## Шаг 3. Как посчитать, сколько упаковок выдержали испытание в каждой партии?

In [ ]:
first.record(3, 2)
second.record(4, 0)
for batch in [first, second]:
    print(batch.label, batch.passed(), "из", batch.tested)
print("Всего годных:", first.passed() + second.passed())

**Обсуждение.** В первой партии испытали восемь упаковок и обнаружили три протечки. У второй все четыре упаковки годны. Метод passed каждый раз вычисляет разность из текущих данных, так что при добавлении протокола отдельный счётчик годных обновлять не требуется.

## Поиск логической ошибки
Почему результат может ввести в заблуждение?

In [ ]:
class DraftBatch:
    def __init__(self):
        self.tested = 5
        self.leaks = 1

    def record(self, tested, leaks):
        self.tested += tested
        if not 0 <= leaks <= tested:
            return False
        self.leaks += leaks
        return True

draft = DraftBatch()
print(draft.record(2, 3))
print(draft.tested, draft.leaks)

Метод сообщил об отказе, но успел увеличить число проверенных с пяти до семи. Отчёт приписал бы партии шесть годных упаковок вместо четырёх. Перенесите изменение обоих счётчиков за проверку всего протокола, как сделано в первой ячейке.

## Измените задачу
В продолжение третьей ячейки добавьте в L-02 два протокола. В первом проверена одна упаковка и обнаружена одна протечка. Во втором проверено ноль упаковок и найдено ноль протечек. Что будет принято и сколько упаковок останутся годными в обеих партиях вместе?

In [ ]:
# Ваше решение

Для проверки: Первый протокол допустим, показатели L-02 станут 5 проверенных и 1 протечка. Второй вызов приведёт к ValueError и сохранит эти числа. В L-02 по-прежнему 4 годных, в K-17 — 5, вместе — 9.